# Wind Generation Forecasting

This notebook develops machine learning models to forecast wind power generation in Germany using weather, temporal, demand, and renewable energy system features.

The target variable is:

- DE_wind_generation_actual

Wind power is one of Germany's most important renewable energy sources and plays a critical role in the country's energy transition. Accurate forecasting of wind generation supports grid operation, renewable integration, and carbon emissions reduction strategies.

The objective of this notebook is to identify the most effective machine learning model for predicting wind generation and to understand the key drivers influencing wind power output.

## Data Preparation and Chronological Train-Test Split

Because wind generation is time-dependent, the dataset is split chronologically rather than randomly. The first 80% of observations are used for training, while the final 20% are reserved for testing.

This approach better reflects a real forecasting scenario, where models are trained on historical data and evaluated on future unseen observations.

In [14]:
import pandas as pd
import numpy as np

# Load model-ready dataset
df = pd.read_csv("../data/processed/model_ready_features.csv")

print("Dataset shape:", df.shape)

# Ensure chronological ordering
df = df.sort_values("utc_timestamp").reset_index(drop=True)

# Define target variable
target_col = "DE_wind_generation_actual"

y = df[target_col]

# Define feature matrix
X = df.drop(
    columns=[
        target_col,
        "DE_LU_price_day_ahead",
        'renewable_generation',
        'renewable_share_pct',
        "utc_timestamp",
        "time"
    ],
    errors="ignore"
)

# Chronological 80/20 split
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTraining set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nMissing values in X:", X.isnull().sum().sum())

Dataset shape: (50232, 31)
Feature matrix shape: (50232, 25)
Target shape: (50232,)

Training set: (40185, 25)
Test set: (10047, 25)

Missing values in X: 0


The dataset was successfully divided into chronological training and test sets. This avoids data leakage from the future into the training process and provides a more realistic evaluation of wind generation forecasting performance.

## Baseline Forecast

Before applying machine learning models, a simple baseline forecast is established.

The baseline predictor uses the mean wind generation observed in the training dataset and predicts this constant value for all observations in the test set.

This benchmark provides a reference point against which the performance of more sophisticated forecasting models can be evaluated.

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_prediction = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline MAE :", round(baseline_mae, 4))
print("Baseline RMSE:", round(baseline_rmse, 4))
print("Baseline R²  :", round(baseline_r2, 4))

Baseline MAE : 8334.1209
Baseline RMSE: 11049.1728
Baseline R²  : -0.1096


## Baseline Model Performance

The baseline model predicts the mean wind generation observed in the training dataset for every observation in the test set.

| Metric | Value |
|----------|----------:|
| MAE | 8334.12 |
| RMSE | 11049.17 |
| R² | -0.1096 |

The baseline model exhibits poor predictive performance, as expected. The negative R² value indicates that simply predicting the mean wind generation performs worse than using the actual average of the test data as a predictor.

These results establish a benchmark against which more sophisticated machine learning models can be evaluated. Any useful forecasting model should significantly reduce prediction errors and achieve a positive R² score relative to this baseline.

## Linear Regression Model

Linear Regression is used as the first machine learning model for wind generation forecasting.

This model assumes a linear relationship between the predictor variables and wind power generation. Although wind generation is influenced by complex meteorological processes, Linear Regression provides a useful benchmark and helps assess whether simple linear relationships are sufficient to explain variations in wind output.

The model is evaluated using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)

Its performance will be compared against the baseline model and more advanced machine learning approaches.

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train model
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

# Metrics
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression MAE :", round(lr_mae, 4))
print("Linear Regression RMSE:", round(lr_rmse, 4))
print("Linear Regression R²  :", round(lr_r2, 4))

Linear Regression MAE : 4050.6753
Linear Regression RMSE: 5223.4037
Linear Regression R²  : 0.752


## Linear Regression Results

| Metric | Value |
|----------|----------:|
| MAE | 4050.68 |
| RMSE | 5223.40 |
| R² | 0.7520 |

The Linear Regression model substantially outperformed the baseline forecast, reducing both MAE and RMSE by more than 50%.

An R² value of 0.752 indicates that approximately 75% of the variation in wind generation can be explained by the selected predictor variables. This demonstrates that weather, temporal, and energy system features contain significant predictive information regarding wind power output.

Despite this improvement, a considerable portion of the variance remains unexplained, suggesting the presence of nonlinear relationships that cannot be fully captured by a linear model. More advanced machine learning approaches such as Random Forest and XGBoost are therefore evaluated in subsequent sections.

## Random Forest Model

Random Forest is an ensemble machine learning algorithm that combines the predictions of multiple decision trees.

Unlike Linear Regression, Random Forest can model nonlinear relationships and interactions among variables without requiring explicit assumptions about the underlying data distribution. This capability makes it particularly suitable for wind generation forecasting, where meteorological conditions often influence power output in complex ways.

The Random Forest model is trained using the same feature set and chronological train-test split as the previous models. Its performance is evaluated using MAE, RMSE, and R² to determine whether nonlinear modelling improves forecasting accuracy.

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train model
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest MAE :", round(rf_mae, 4))
print("Random Forest RMSE:", round(rf_rmse, 4))
print("Random Forest R²  :", round(rf_r2, 4))

Random Forest MAE : 4021.2774
Random Forest RMSE: 5238.7465
Random Forest R²  : 0.7506


## Random Forest Results

| Metric | Value |
|----------|----------:|
| MAE | 4021.28 |
| RMSE | 5238.75 |
| R² | 0.7506 |

The Random Forest model achieved performance comparable to Linear Regression but did not produce a meaningful improvement in predictive accuracy.

While the MAE decreased slightly, both RMSE and R² were marginally worse than those obtained using Linear Regression. This suggests that the current feature set may already capture much of the underlying relationship between the predictors and wind generation through relatively simple patterns.

The results indicate that increasing model complexity does not automatically lead to better forecasting performance. Further improvements may require more informative features rather than a more complex learning algorithm.

The next step is to evaluate XGBoost, a gradient boosting model that often performs better than Random Forest on structured tabular datasets.